# 10 — Comparar contra PCT (Zhao ICLR 2022)

Lee `09_homologar_pct.json`. Primero la **tabla de comparabilidad**: qué
columnas están en ambos lados bajo el mismo protocolo. Después el
veredicto.

Con este `09`: hay decisión en **factibilidad** (bin cerrado 2000 mm).
**No** hay Uti. vs Uti. de Table 1 (PCT no es factible; sin plan LFS
no hay volumen dentro del bin). Si se ignora la contención, el volumen
de los 26 ítems empata; eso no decide.

Un pedido: no es ranking BED-BPP ni Σalgo.

In [1]:
from pathlib import Path
import json
import sys

HERE = Path.cwd().resolve()
if HERE.name == "notebooks":
    ML_ROOT = HERE.parent
else:
    ML_ROOT = HERE if (HERE / "src_ml").is_dir() else Path("/home/edmundo/packing-services/online_policy_ml")
for p in (ML_ROOT / "src_ml", ML_ROOT.parent / "src"):
    s = str(p)
    if s not in sys.path:
        sys.path.insert(0, s)

from bootstrap import setup
setup()
from bedbpp_eval import tabla_comparacion, veredicto_zhao
from evaluate import write_report
from paths import REPORTS_DIR

doc = json.loads((REPORTS_DIR / "09_homologar_pct.json").read_text(encoding="utf-8"))
assert "veredicto_zhao" in doc, "Reejecuta el 09: el JSON viejo no homologa en bin cerrado Zhao."
nuestro = {r["order_id"]: r for r in doc["nuestro"]}
pct = doc.get("pct") or {}
shared = sorted(set(nuestro) & set(pct))
assert shared, "No hay IDs compartidos. No se compara."

tablas = {oid: tabla_comparacion(nuestro[oid], pct[oid]) for oid in shared}
veredictos = {oid: veredicto_zhao(nuestro[oid], pct[oid]) for oid in shared}

for oid in shared:
    print(f"=== {oid} comparabilidad ===")
    for fila in tablas[oid]["filas"]:
        marca = "DECIDE" if fila.get("decide") else ("sí" if fila.get("comparable") else "no")
        print(
            f"  [{marca:6}] {fila['columna']}: nuestro={fila['nuestro']} | pct={fila['pct']}"
            + (f"  ({fila['nota']})" if fila.get("nota") else "")
        )
    print("columna decisoria:", tablas[oid]["columna_decisoria"])
    print("Uti. head-to-head:", tablas[oid]["uti_head_to_head"])
    print(veredictos[oid]["frase"])
    print()

if len(shared) == 1:
    frase = veredictos[shared[0]]["frase"]
    lado = veredictos[shared[0]]["lado"]
else:
    lados = {veredictos[oid]["lado"] for oid in shared}
    lado = lados.pop() if len(lados) == 1 else "mixto"
    frase = " ".join(veredictos[oid]["frase"] for oid in shared)

write_report(
    REPORTS_DIR / "10_comparar_pct.json",
    {
        "marco": "Zhao ICLR 2022, bin cerrado 1200×800×2000 mm",
        "shared": shared,
        "comparabilidad": tablas,
        "veredictos": veredictos,
        "lado": lado,
        "columna_decisoria": "feasible",
        "uti_head_to_head": all(tablas[oid]["uti_head_to_head"] for oid in shared),
        "veredicto": frase,
        "alcance": "un pedido; no ranking BED-BPP; no Σalgo; no Table 1 10×10×10",
    },
)

=== 00100408 comparabilidad ===
  [sí    ] tarea / pedido / bin: nuestro=O3DBP-1-1, 00100408, 1200×800×2000 mm | pct=O3DBP, 00100408, europalé (Kagerer corrió PCT aquí)
  [DECIDE] feasible (contención Zhao): nuestro=True | pct=False  (en Zhao una caja fuera del bin no es solución Table 1)
  [no    ] Uti. Zhao (V_dentro / V_bin): nuestro=0.646376 | pct=None  (PCT sin Uti. en bin cerrado (plan LFS o hn>2 m))
  [no    ] Num. Zhao (ítems dentro): nuestro=26 | pct=None
  [sí    ] Uti. si se ignora contención: nuestro=0.646376 | pct=0.646235  (mismo volumen de 26 ítems ≈ empate; no es Zhao)
  [sí    ] ηutil Kagerer: nuestro=0.656219 | pct=0.614  (ortoedro A×hn; no es Uti. del paper)
  [sí    ] hn (m): nuestro=1.97 | pct=2.105
columna decisoria: feasible
Uti. head-to-head: False
En el pedido homologado, O3DBP p=1 s=1, bin 1200×800×2000 mm (Zhao ICLR 2022: Uti./Num., contención), `mlp_v1_p1s1_ppo.pt` supera PCT. nuestra solución es factible en el bin 1200×800×2000 mm; la de PCT publicada no (h